# Prediction With the Trained Simple RNN

Load `simple_rnn_imdb.keras` (saved by the project notebook) and classify new reviews.

In [6]:
import re
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import pad_sequences

max_features = 10000   # same vocabulary size as training
max_len = 500          # same review length as training

## 1. Load the word index and the model

In [7]:
word_index = imdb.get_word_index()
reverse_word_index = {index: word for word, index in word_index.items()}

model = load_model('simple_rnn_imdb.keras')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,939,077 (15.03 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,626,052 (10.02 MB)

## 2. Look at the learned weights

Embedding table, then Wi (kernel), Wh (recurrent_kernel), the bias, and Wo with its bias.

In [10]:
# model.weights
for w in model.weights:
    print(w.path, tuple(w.shape))

embedding/embeddings (10000, 128)
simple_rnn/simple_rnn_cell/kernel (128, 128)
simple_rnn/simple_rnn_cell/recurrent_kernel (128, 128)
simple_rnn/simple_rnn_cell/bias (128,)
dense/kernel (128, 1)
dense/bias (1,)


## 3. Helper functions

`preprocess_text` must turn text into ids **exactly** like the training data: lowercase, no punctuation, start token 1, unknown word 2, real words = index + 3, only ids below 10,000, padded to 500.

In [11]:
def decode_review(encoded_review):
    return ' '.join(reverse_word_index.get(i - 3, '?') for i in encoded_review)


def preprocess_text(text):
    words = re.findall(r"[a-z0-9']+", text.lower())   # lowercase, drop punctuation
    encoded = [1]                                      # 1 = start of review
    for word in words:
        index = word_index.get(word)
        if index is None or index + 3 >= max_features:
            encoded.append(2)                          # 2 = unknown word
        else:
            encoded.append(index + 3)                  # real words start at 4
    return pad_sequences([encoded], maxlen=max_len)

In [15]:
# check: what does the model actually see?
sample = preprocess_text('The songs were beautiful!')
print(sample)
print(decode_review(sample[0]))

[[  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0   0   0   0   0 

## 4. Prediction function

In [21]:
def predict_sentiment(review):
    score = float(model.predict(preprocess_text(review), verbose=0)[0][0])
    sentiment = 'Positive' if score > 0.5 else 'Negative'
    return sentiment, score


predict_sentiment("The songs were beautiful and the movie was great, the actors did great acting and i will recommend it to everyone")

('Positive', 0.9545668959617615)

## 5. Try some reviews

In [22]:
reviews = [
    'The songs were beautiful and the lead pair acted brilliantly. A must watch with family.',
    'A boring film. The story made no sense and I left halfway.',
    'The movie was scary and I did not like it, but the critics loved it.',
]

for review in reviews:
    sentiment, score = predict_sentiment(review)
    print(f'{sentiment:8} {score:.3f}  {review}')

Positive 0.925  The songs were beautiful and the lead pair acted brilliantly. A must watch with family.
Negative 0.207  A boring film. The story made no sense and I left halfway.
Positive 0.859  The movie was scary and I did not like it, but the critics loved it.
